# LoRA Fine-Tuning on Text Data with a Small Model

In [1]:
!pip install bitsandbytes accelerate loralib transformers datasets peft trl safetensors flash-linear-attention

    100% |████████████████████████████████| 103.2MB 434kB/s eta 0:00:011   18% |█████▉                          | 18.9MB 82.0MB/s eta 0:00:02    58% |██████████████████▋             | 60.2MB 54.2MB/s eta 0:00:01    66% |█████████████████████▎          | 68.6MB 46.6MB/s eta 0:00:01    72% |███████████████████████         | 74.5MB 344kB/s eta 0:01:24    92% |█████████████████████████████▌  | 95.2MB 61.8MB/s eta 0:00:01
  Installing build dependencies ... done
    Complete output from command python setup.py egg_info:
    Traceback (most recent call last):
      File "<string>", line 1, in <module>
      File "/private/var/folders/2h/3cwp6jzs5yb1dtj9t6489mhw0000gn/T/pip-install-9AsroR/bitsandbytes/setup.py", line 21
        name=f"bitsandbytes",
                           ^
    SyntaxError: invalid syntax
    
    ----------------------------------------
Command "python setup.py egg_info" failed with error code 1 in /private/var/folders/2h/3cwp6jzs5yb1dtj9t6489mhw0000gn/T/pip-install-9Asr

## Load Pre-trained Model

In [2]:
import torch
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, GenerationConfig, pipeline

/Users/parnnoi/Coding/DL/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1117 17:05:21.758000 66069 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
from IPython.display import display_markdown, Latex, display
def print_markdown(content):
    if isinstance(content, str) and "\\" in content:
        display(Latex(content))
    else:
        display_markdown(content, raw=True)

In [4]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B-Base"
OUTPUT_DIR = "./lora-demo"

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Model: {MODEL_NAME}")

Device: mps
Model: Qwen/Qwen3.5-0.8B-Base


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype
).to(device)

print(f"Tokenizer pad token: {tokenizer.pad_token}")

KeyboardInterrupt: 

## Get baseline generation

In [ ]:
import copy
generation_config = copy.deepcopy(model.generation_config)

In [ ]:
generation_config.max_new_tokens = 500
generation_config.do_sample = True
generation_config.temperature = 0.5

In [ ]:
generator = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer
)

**Question:** Which viruses are best known to cause myocarditis?  
**Expected answer:** Enterovirus, Adenovirus and Coxsackie B viruses.  

In [ ]:
%%time
prompt = "Which viruses are best known to cause myocarditis?"
result = generator(prompt, generation_config=generation_config)

In [ ]:
print_markdown(result[0]["generated_text"])

## Load and prepare dataset

In [9]:
import pandas as pd
import datasets

In [ ]:
dataset = pd.read_json("alpaca_data_cleaned.json")
dataset.shape

In [ ]:
dataset.sample(5)

,instruction,input,output
25716,Describe a method of converting solar energy i...,,One of the most common methods of converting s...
45324,Develop an algorithm to add two numbers together,,Here's an algorithm to add two numbers togethe...
5762,"Imagine you are making an egg sandwich, write ...",,Here is a step-by-step instruction to make an ...
41175,Generate three unique subject lines for an ema...,,"1. ""Get ready to transform your health with ou..."
51252,Give a convincing argument for.,Argument for studying history,Studying history is crucial for a number of re...


### Create a dataset object

In [ ]:
alpaca_dataset = datasets.Dataset.from_pandas(dataset)

### Format dataset for fine-tuning

In [19]:
def format_instruction(example):
    if example["input"].strip():
        text = (
            f"Instruction: {example['instruction']}\n"
            f"Input: {example['input']}\n"
            f"Response: {example['output']}"
        )
    else:
        text = (
            f"Instruction: {example['instruction']}\n"
            f"Response: {example['output']}"
        )
    return {"text": text}

In [ ]:
formatted_dataset = alpaca_dataset.map(format_instruction)

Map: 100%|██████████| 51760/51760 [00:01<00:00, 33625.63 examples/s]


In [21]:
print(formatted_dataset[42]["text"])

Instruction: Explain why the given definition is wrong.
Input: A mole is an animal that lives underground.
Response: While it is true that a mole is an animal that lives underground, this definition is incomplete and could be misunderstood because it does not adequately describe the mole's characteristics or biological classification. Moles belong to the family of mammals called Talpidae, and they are small, burrowing, and insectivorous. They are known for their cylindrical bodies, short limbs, and small eyes and ears. The definition also does not differentiate the animal from the chemical term mole, which refers to a unit of measurement used in chemistry. For these reasons, the given definition of a mole is not entirely accurate or clearly descriptive.


## Setting up LoRA

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

- **r:** The rank of the A and B matrices
- **lora_alpha:** Scaling factor. Usually 2x the rank.
- **target_modules:** The portions of the model we want to optimize with LoRA (depends on model architecture).
- **lora_dropout:** Probability of dropout to helps prevent the model from overfitting.
- **bias:** Whether to train LoRA on bias components as well. Set to `none` means we are training only on weighs component.
- **task_type:** Setting to `CAUSAL_LM` because the specific language model we're using is "causal".

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

### Attach LoRA adapter to the model

In [ ]:
model = get_peft_model(model, lora_config)

### Configure supervised fine-tuning

In [ ]:
training_args = SFTConfig(
    dataset_text_field="text",  
    max_length=512,
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=10,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=20,
    max_steps=100,
    save_strategy="epoch",
    optim="adamw_torch"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args,
    processing_class=tokenizer
)

### Total trainable parameters

In [ ]:
trainer.model.print_trainable_parameters()

### Total training steps (if not using `max_steps`)

In [ ]:
effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
total_steps = ((len(alpaca_dataset)+1) // effective_batch_size) * training_args.num_train_epochs

In [ ]:
print(f"Total training steps: {total_steps}")

## Training LoRA

In [ ]:
trainer.train()

## Save trained model (LoRA Adapter and Tokenizer)

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved LoRA adapter to:", OUTPUT_DIR)

## Reload base model and attach LoRA adapter

In [ ]:
from peft import PeftModel

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32

In [ ]:
# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype
).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_generator = pipeline(
    "text-generation", 
    model=base_model, 
    tokenizer=tokenizer
)

# Base model + LoRA adapter
adapted_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
adapted_generator = pipeline(
    "text-generation",
    model=adapted_model,
    tokenizer=tokenizer,
    device=device
)

## Compare outputs of base and fine-tuned model

In [ ]:
import copy
generation_config = copy.deepcopy(model.generation_config)

In [ ]:
generation_config.max_new_tokens = 200
generation_config.do_sample = True
generation_config.temperature = 0.5

In [ ]:
questions = ["Which viruses are best known to cause myocarditis?",
             "What is the role of the blood-brain barrier?",
             "From which cell type is leptin secreted?",
             "What is the causative agent of the Panama disease affecting bananas?"
            ]

In [ ]:
for q in questions:
    print(f"\n{"="*150}")
    print(f"Question: {q}")

    base_out = base_generator(q, generation_config=generation_config)[0]["generated_text"]
    lora_out = adapted_generator(q, generation_config=generation_config)[0]["generated_text"]

    print("\n--- Base model ---")
    print(base_out)

    print("\n--- LoRA fine-tuned model ---")
    print(lora_out)

---